# A.2: Training with LoRA

Train only the adapters on the synthetic tasks, with the same data and steps as full fine-tuning, for several ranks.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From A.1 A synthetic instruction dataset
import random, math, time

import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

WORDS = ("apple river stone cloud garden window candle forest pencil mirror ocean silver "
         "basket rocket island winter summer planet dragon castle butter meadow violin "
         "tiger lemon piano wallet bridge jacket carpet rabbit button harbor lantern "
         "orange parrot saddle ticket valley wizard anchor bottle circle desert engine "
         "feather guitar hammer jungle kettle ladder marble needle pepper puzzle ribbon "
         "shadow thunder tunnel velvet walnut yellow zipper blanket cactus dolphin").split()

def sentence(rng):
    return " ".join(rng.sample(WORDS, rng.randint(3, 6)))

TASKS = {  # name: (phrasings; the last one is held out, answer function, input maker)
    "upper": (["Convert to uppercase: {x}", "Write {x} in capital letters.",
               "Uppercase this word: {x}", "Rewrite the word {x} using only capital letters."],
              lambda x: x.upper(), lambda r: r.choice(WORDS)),
    "first": (["What is the first letter of '{x}'?", "Give the first letter of the word {x}.",
               "Which letter does '{x}' start with?", "Tell me the initial letter of {x}."],
              lambda x: x[0], lambda r: r.choice(WORDS)),
    "repeat": (["Repeat the word {x} three times.", "Say {x} three times, separated by spaces.",
                "Write '{x}' 3 times.", "Output the word {x} thrice."],
               lambda x: " ".join([x] * 3), lambda r: r.choice(WORDS)),
    "count": (["How many words are in this text: {x}", "Count the words: {x}",
               "Number of words in '{x}'?", "Tell me how many words this has: {x}"],
              lambda x: str(len(x.split())), sentence),
    "last": (["What is the last word of: {x}", "Give the final word of '{x}'.",
              "Which word comes last in: {x}", "Name the word at the end of: {x}"],
             lambda x: x.split()[-1], sentence),
    "add": (["What is {a} + {b}?", "Add {a} and {b}.", "Compute {a} plus {b}.",
             "What do you get when you add {a} to {b}?"],
            None, lambda r: (r.randint(10, 49), r.randint(1, 9))),
}

def make_example(task, rng, held_out=False):
    phrasings, answer, make = TASKS[task]
    p = phrasings[-1] if held_out else rng.choice(phrasings[:-1])
    x = make(rng)
    if task == "add":
        return p.format(a=x[0], b=x[1]), str(x[0] + x[1])
    return p.format(x=x), answer(x)

rng = random.Random(0)

train = [make_example(t, rng) for _ in range(100) for t in TASKS]

test_seen = [make_example(t, rng) for _ in range(5) for t in TASKS]

test_new = [make_example(t, rng, held_out=True) for _ in range(5) for t in TASKS]

print(len(train), "training examples,", len(test_seen), "test (seen phrasings),",
      len(test_new), "test (held-out phrasings)")

print("seen-phrasing test examples that also occur in the training set:",
      sum(ex in set(train) for ex in test_seen))

for q, a in train[:6]:
    print(f"  {q!r} -> {a!r}")

# From A.2 Tokenizing with loss masks and padding
tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")

IM_START, IM_END = "<|im_start|>", "<|im_end|>"

def prompt_text(q):
    return f"{IM_START}user\n{q}{IM_END}\n{IM_START}assistant\n"

def encode(q, a, mask_prompt=True):
    p = tok(prompt_text(q), add_special_tokens=False).input_ids
    r = tok(a + IM_END, add_special_tokens=False).input_ids
    labels = ([-100] * len(p) if mask_prompt else p) + r
    return p + r, labels

def collate(batch):
    n = max(len(ids) for ids, _ in batch)
    ids = torch.full((len(batch), n), tok.pad_token_id or 0)
    labels = torch.full((len(batch), n), -100)
    attn = torch.zeros((len(batch), n), dtype=torch.long)
    for i, (x, y) in enumerate(batch):
        ids[i, :len(x)], labels[i, :len(y)], attn[i, :len(x)] = torch.tensor(x), torch.tensor(y), 1
    return ids, labels, attn

ids, labels = encode(*train[0])

print("prompt + answer tokens:", len(ids), "| trained:", sum(l != -100 for l in labels))

print("trained:", [tok.decode([t]) for t, l in zip(ids, labels) if l != -100])

# From A.4 Evaluation: exact match and forgetting
text = open("shakespeare.txt", encoding="utf-8").read()

held = tok(text[-20000:], add_special_tokens=False).input_ids

blocks = [held[i:i + 128] for i in range(0, 20 * 128, 128)]

@torch.no_grad()
def accuracy(model, data):
    model.eval()
    correct = 0
    for q, a in data:
        ids = torch.tensor([tok(prompt_text(q), add_special_tokens=False).input_ids])
        out = model.generate(ids, max_new_tokens=12, do_sample=False,
                             eos_token_id=tok.convert_tokens_to_ids(IM_END),
                             pad_token_id=tok.convert_tokens_to_ids(IM_END))
        pred = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()
        correct += pred == a
    return correct / len(data)

@torch.no_grad()
def perplexity(model):
    model.eval()
    x = torch.tensor(blocks)
    return math.exp(model(x, labels=x).loss.item())

def report(name, model):
    print(f"{name:34s} seen {accuracy(model, test_seen):.2f}  held-out phrasing "
          f"{accuracy(model, test_new):.2f}  Shakespeare ppl {perplexity(model):.1f}")

# From A.1 A LoRA layer from scratch
import torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.scale = alpha / r
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.A.T @ self.B.T)

    def merged(self):
        """A plain nn.Linear with the update folded into the weight."""
        out = nn.Linear(self.base.in_features, self.base.out_features,
                        bias=self.base.bias is not None)
        with torch.no_grad():
            out.weight.copy_(self.base.weight + self.scale * self.B @ self.A)
            if self.base.bias is not None:
                out.bias.copy_(self.base.bias)
        return out

def add_lora(model, targets=("q_proj", "k_proj", "v_proj", "o_proj"), r=8, alpha=16):
    for p in model.parameters():
        p.requires_grad_(False)
    for layer in model.model.layers:
        attn = layer.self_attn
        for name in targets:
            setattr(attn, name, LoRALinear(getattr(attn, name), r, alpha))
    return model


In [ ]:
def finetune_lora(r, steps=120, batch_size=8, lr=1e-3, warmup=10, seed=0):
    torch.manual_seed(seed)
    m = add_lora(AutoModelForCausalLM.from_pretrained(
        "HuggingFaceTB/SmolLM2-135M", dtype=torch.float32), r=r, alpha=2 * r)
    data = [encode(q, a) for q, a in train]
    params = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warmup, (steps - s) / (steps - warmup)))
    order = random.Random(seed)
    m.train()
    for step in range(steps):
        ids, labels, attn = collate(order.sample(data, batch_size))
        loss = m(input_ids=ids, attention_mask=attn, labels=labels).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step(); sched.step(); opt.zero_grad()
    return m, sum(p.numel() for p in params)

start = time.time()
adapters = {}
for r in [2, 8, 32]:
    m, n = finetune_lora(r)
    adapters[r] = m
    report(f"LoRA r={r} ({n:,} trainable)", m)
print(f"(three runs: {(time.time() - start) / 60:.0f} minutes on 4 CPU threads)")


**Expected output**

Output:

```text
LoRA r=2 (230,400 trainable)       seen 0.00  held-out phrasing 0.00  Shakespeare ppl 56.4
LoRA r=8 (921,600 trainable)       seen 0.57  held-out phrasing 0.43  Shakespeare ppl 74.1
LoRA r=32 (3,686,400 trainable)    seen 0.73  held-out phrasing 0.50  Shakespeare ppl 322.8
(three runs: 6 minutes on 4 CPU threads)
```
